## Eksport HPO (JSONL) i wczytanie tłumaczenia (pandas)

Ten notatnik:
- pobiera z bazy wiersze z `hpo_terms`, gdzie `label_pl` jest puste,
- zapisuje paczkę do `output/hpo_missing_label_pl.jsonl`,
- wczytuje odpowiedź LLM z pliku **`input/tłumaczenie.jsonl`** (pandas),
- scala po `hpo_id` i (opcjonalnie) zapisuje `label_pl` do bazy.

Wymagane: `DATABASE_URL` w `backend/.env` (jak dla backendu).

Uwaga: nie wklejaj do czata żadnych danych z `DATABASE_URL` (to sekret).


In [5]:
from __future__ import annotations

import csv
import sys
from pathlib import Path

from sqlalchemy import select
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

# Katalog backend/ — tam jest pakiet `app/`; działa niezależnie od CWD (np. scripts/v2).
_cwd = Path.cwd().resolve()
BACKEND_DIR = next(p for p in [_cwd, *_cwd.parents] if (p / "app").is_dir())
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from app.config import settings
from app.core.database import database_url
from app.models.v2.hpo import HpoTerm

print("database_url driver:", database_url.split("://", 1)[0])
print("BACKEND_DIR:", BACKEND_DIR)
print("Czy jest skonfigurowane połączenie (settings.database_url)?:", bool(settings.database_url))

database_url driver: postgresql+asyncpg
BACKEND_DIR: /home/mario/workspace/zebrapoint/backend
Czy jest ustawiona DATABASE_URL w env?: True


In [31]:
import asyncio
import json


def _default_output_dir() -> Path:
    out_dir = Path.cwd() / "output"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def _default_tsv_path() -> Path:
    return _default_output_dir() / "hpo_missing_label_pl.tsv"


def _default_jsonl_path() -> Path:
    return _default_output_dir() / "hpo_missing_label_pl.jsonl"


async def _fetch_missing_hpo_rows(*, limit: int) -> list[tuple[str, str]]:
    if limit < 1:
        raise ValueError("limit musi być >= 1")

    engine = create_async_engine(database_url, echo=False)
    session_factory = async_sessionmaker(engine, expire_on_commit=False)

    try:
        async with session_factory() as session:
            stmt = (
                select(HpoTerm.hpo_id, HpoTerm.label_en)
                .where((HpoTerm.label_pl.is_(None)) | (HpoTerm.label_pl == ""))
                .order_by(HpoTerm.hpo_id)
                .limit(limit)
            )
            result = await session.execute(stmt)
            return [(hpo_id, label_en or "") for (hpo_id, label_en) in result.all()]
    finally:
        await engine.dispose()


async def export_missing_hpo_to_tsv(
    *,
    limit: int = 200,
    out_path: Path | None = None,
    include_header: bool = False,
) -> Path:
    """Eksportuje `hpo_id` i `label_en` dla terminów bez `label_pl` do TSV."""

    out_path = out_path or _default_tsv_path()
    rows = await _fetch_missing_hpo_rows(limit=limit)

    with out_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t", lineterminator="\n")
        if include_header:
            w.writerow(["hpo_id", "label_en"])
        w.writerows(rows)

    print(f"Zapisano TSV: {out_path} (wierszy: {len(rows)})")
    return out_path


async def export_missing_hpo_to_jsonl(*, limit: int = 200, out_path: Path | None = None) -> Path:
    """Eksportuje rekordy do JSONL (jedna linia = jeden obiekt JSON)."""

    out_path = out_path or _default_jsonl_path()
    rows = await _fetch_missing_hpo_rows(limit=limit)

    with out_path.open("w", encoding="utf-8", newline="") as f:
        for hpo_id, label_en in rows:
            f.write(json.dumps({"hpo_id": hpo_id, "label_en": label_en}, ensure_ascii=False) + "\n")

    print(f"Zapisano JSONL: {out_path} (wierszy: {len(rows)})")
    return out_path


# Ustaw ile rekordów chcesz wysłać do czata (praktycznie: 200–400; 1000 bywa za duże na limity)
LIMIT = 1000

# Wybierz format eksportu:
# out_path = await export_missing_hpo_to_tsv(limit=LIMIT, include_header=False)
out_path = await export_missing_hpo_to_jsonl(limit=LIMIT)

out_path

Zapisano JSONL: /home/mario/workspace/zebrapoint/backend/scripts/v2/output/hpo_missing_label_pl.jsonl (wierszy: 1000)


PosixPath('/home/mario/workspace/zebrapoint/backend/scripts/v2/output/hpo_missing_label_pl.jsonl')

In [25]:
# Podgląd eksportu (pandas)

import pandas as pd

df_export = pd.read_json(out_path, lines=True)
print("Liczba wierszy:", len(df_export))
print(df_export.head(30).to_string(index=False))


Liczba wierszy: 500
    hpo_id                                                        label_en
HP:0000649                         Abnormality of visual evoked potentials
HP:0000650 Abnormal amplitude of pattern reversal visual evoked potentials
HP:0000651                                                        Diplopia
HP:0000652                                           Lower eyelid coloboma
HP:0000653                                                Sparse eyelashes
HP:0000654   Decreased light- and dark-adapted electroretinogram amplitude
HP:0000656                                                       Ectropion
HP:0000657                                              Oculomotor apraxia
HP:0000658                                                  Eyelid apraxia
HP:0000659                                                  Peters anomaly
HP:0000660                                               Lipemia retinalis
HP:0000661                        Palpebral fissure narrowing on adduction
HP:00

## Lepszy format dla LLM: JSONL (polecam)

TSV jest OK, ale **JSONL** zwykle jest dla modeli czytelniejszy i mniej podatny na „zgubienie kolumn”.

- Każda linia to osobny JSON, np.:
  - `{ "hpo_id": "HP:0000118", "label_en": "Phenotypic abnormality" }`

Poproś LLM, żeby zwrócił JSONL w tym samym układzie, ale z dodanym polem `label_pl`.

### Prompt + plik `input/tłumaczenie.jsonl`

1. Wyślij do czata zawartość `output/hpo_missing_label_pl.jsonl`.
2. Odpowiedź zapisz jako **`input/tłumaczenie.jsonl`** (jedna linia = jeden JSON: `hpo_id`, `label_pl`).
3. Uruchom komórkę **„Wczytanie tłumaczenia”** — odczyt przez **pandas**.

Przykładowy prompt:

```text
Przetłumacz na polski nazwy HPO.

Wejście: JSONL (hpo_id, label_en).
Wyjście: JSONL — tyle samo rekordów, ta sama kolejność hpo_id.
Każda linia: {"hpo_id": "HP:...", "label_pl": "..."}

Zasady:
- Nie zmieniaj hpo_id.
- Nie dodawaj ani nie usuwaj linii.
- Bez markdown w odpowiedzi — tylko czysty JSONL.

[TU WKLEJ PLIK .jsonl]
```

Jeśli model „sklei" wiele `{...}` w jednej linii, komórka spróbuje to rozłożyć — i tak bezpieczniej jest jeden obiekt na linię.


In [35]:
import json
import re
from io import StringIO
from pathlib import Path

import pandas as pd

INPUT_DIR = Path.cwd() / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
TRANSLATION_PATH = INPUT_DIR / "tłumaczenie.jsonl"


def _strip_markdown_fence(text: str) -> str:
    t = text.strip()
    m = re.match(r"^```(?:json|JSON)?\s*\n?", t)
    if m:
        t = t[m.end() :]
    t = re.sub(r"\s*```\s*$", "", t)
    return t.strip()


def load_translation_dataframe(path: Path) -> pd.DataFrame:
    """Wczytuje JSONL z odpowiedzią LLM; obsługuje też kilka obiektów JSON w jednej linii."""
    if not path.exists():
        raise FileNotFoundError(
            f"Brak pliku {path}. Zapisz odpowiedź czata jako input/tłumaczenie.jsonl"
        )
    raw = path.read_text(encoding="utf-8")
    raw = _strip_markdown_fence(raw)
    if not raw.strip():
        raise ValueError("Plik tłumaczenia jest pusty")

    try:
        df = pd.read_json(StringIO(raw), lines=True)
    except ValueError:
        dec = json.JSONDecoder()
        rows: list[dict] = []
        idx = 0
        while idx < len(raw):
            while idx < len(raw) and raw[idx].isspace():
                idx += 1
            if idx >= len(raw):
                break
            obj, end = dec.raw_decode(raw, idx)
            idx = end
            if isinstance(obj, dict):
                rows.append(obj)
            else:
                raise ValueError("Oczekiwane obiekty JSON z polami hpo_id i label_pl")
        df = pd.DataFrame(rows)

    need = {"hpo_id", "label_pl"}
    if not need.issubset(set(df.columns)):
        raise ValueError(f"Brakuje kolumn {need - set(df.columns)} — jest: {list(df.columns)}")

    out = df[["hpo_id", "label_pl"]].copy()
    out["hpo_id"] = out["hpo_id"].astype(str).str.strip()
    out["label_pl"] = out["label_pl"].astype(str).str.strip()
    out = out[out["hpo_id"].ne("") & out["label_pl"].ne("")]
    if out.empty:
        raise ValueError("Po oczyszczeniu nie zostało żadnego wiersza z hpo_id i label_pl")
    dup = out["hpo_id"].duplicated(keep=False)
    if dup.any():
        bad = out.loc[dup, "hpo_id"].unique()[:20].tolist()
        raise ValueError(f"Zduplikowane hpo_id w tłumaczeniu (przykłady): {bad}")
    return out


export_df = pd.read_json(out_path, lines=True)
tr_df = load_translation_dataframe(TRANSLATION_PATH)

merged = export_df[["hpo_id"]].merge(tr_df, on="hpo_id", how="left")
missing = merged["label_pl"].isna() | (merged["label_pl"].astype(str).str.strip() == "")
if missing.any():
    bad = merged.loc[missing, "hpo_id"].head(20).tolist()
    raise ValueError(
        "Brakuje label_pl dla części hpo_id z eksportu. Przykłady: " + ", ".join(bad)
    )

parsed = list(zip(merged["hpo_id"].astype(str), merged["label_pl"].astype(str)))
print("Eksport (wierszy):", len(export_df))
print("Tłumaczenie (wierszy):", len(tr_df))
print("Połączone (gotowe do zapisu):", len(parsed))
print("Plik tłumaczenia:", TRANSLATION_PATH.resolve())


Eksport (wierszy): 1000
Tłumaczenie (wierszy): 1001
Połączone (gotowe do zapisu): 1000
Plik tłumaczenia: /home/mario/workspace/zebrapoint/backend/scripts/v2/input/tłumaczenie.jsonl


In [36]:
from sqlalchemy import update


async def apply_label_pl_updates(*, updates: list[tuple[str, str]], dry_run: bool = True) -> int:
    """Aktualizuje hpo_terms.label_pl na podstawie listy (hpo_id, label_pl)."""

    if not updates:
        return 0

    engine = create_async_engine(database_url, echo=False)
    session_factory = async_sessionmaker(engine, expire_on_commit=False)

    try:
        async with session_factory() as session:
            changed = 0
            for hpo_id, label_pl in updates:
                if dry_run:
                    continue
                stmt = update(HpoTerm).where(HpoTerm.hpo_id == hpo_id).values(label_pl=label_pl)
                r = await session.execute(stmt)
                changed += int(r.rowcount or 0)
            if not dry_run:
                await session.commit()

        return changed
    finally:
        await engine.dispose()


print("Liczba wierszy do aktualizacji:", len(parsed))

DRY_RUN = False
if DRY_RUN:
    print("DRY RUN — nic nie zapisuję. Ustaw DRY_RUN=False aby zapisać do bazy.")
changed = await apply_label_pl_updates(updates=parsed, dry_run=DRY_RUN)
if not DRY_RUN:
    print("Zaktualizowano wierszy:", changed)


Liczba wierszy do aktualizacji: 1000
Zaktualizowano wierszy: 1000
